In [26]:
import requests
import pandas as pd

url = "http://localhost:8000/upload_config_from_csv/1/"

input_csv = "northwest_phi.csv"

files = {"file": ("all_tables.csv", open(input_csv, "rb"), "text/csv")}

response = requests.post(url, files=files)

print(response.status_code, response.json())

200 {'success': True, 'message': 'Configuration uploaded successfully.'}


In [27]:
df = pd.read_csv(input_csv)
notes_table = list(set(df[df['DE_IDENTIFICATION_RULE'] == 'NOTES']['TABLE_NAME'].to_list()))
len(notes_table)

134

In [28]:
no_notes_df = df[~df['TABLE_NAME'].isin(notes_table)]
no_notes_table = list(set(no_notes_df['TABLE_NAME'].to_list()))
len(no_notes_table)

377

In [29]:
static_df = no_notes_df[no_notes_df['DE_IDENTIFICATION_RULE'] == 'STATIC_OFFSET']
static_tables = list(set(static_df['TABLE_NAME'].to_list()))
len(static_tables)

56

In [1]:
import os
import django
import sys
# Set up Django environment
sys.path.append('/Users/neurodiscoveryai/Desktop/Soorykant/deidentification/deIdentification/')
os.environ.setdefault("DJANGO_SETTINGS_MODULE", "deIdentification.settings")
os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"
django.setup()

from nd_api.views.de_identification_task import create_deidentification_task
from nd_api.models import DbDetailsModel, TableDetailsModel, IgnoreRowsDeIdentificaiton
from worker.models import Task, Chain
from django.contrib.auth.models import User
from keycloakauth.rolemodel import RoleModel, get_default_permissions
from nd_scripts.create_account import create_account
from nd_api.views.db_views import create_stats_generation_tasks
from core.process.main import start_de_identification_for_table
from nd_api.views.de_identification_task import create_deidentification_task
from nd_api.views.db_views import run_stats_generation_task

def clean_db():
    RoleModel.objects.all().delete()
    User.objects.all().delete()
    DbDetailsModel.objects.all().delete()
    Chain.objects.all().delete()

In [2]:
def fix_reference_value(table_id):
    table = TableDetailsModel.objects.get(id=table_id)
    
    pid_col = [
        col['column_name'] for col in table.table_details_for_ui['columns_details'] if col["de_identification_rule"] == "PATIENT_ID"
    ]
    enc_col = [
        col['column_name'] for col in table.table_details_for_ui['columns_details'] if col["de_identification_rule"] == "ENCOUNTER_ID"
    ]
    print(pid_col)
    print(enc_col)
    patient_id = pid_col[0] if len(pid_col)>0 else None
    enc_id = enc_col[0] if len(enc_col)>0 else None
    table.table_details_for_ui['reference_enc_id_column'] = enc_id
    table.table_details_for_ui['reference_patient_id_column'] = patient_id
    table.save()

In [64]:
TableDetailsModel.objects.filter(table_status=0).count()

7619

In [245]:
TableDetailsModel.objects.filter(table_status=1).count()

2

In [243]:
TableDetailsModel.objects.filter(table_status=2).count()

508

In [246]:
TableDetailsModel.objects.filter(table_status=1).values_list('table_name', 'id')

<QuerySet [('pnattachments', 5161), ('progressnotes', 5379)]>

In [233]:
TableDetailsModel.objects.filter(table_status=3).values_list('table_name', 'id')

<QuerySet [('progressnotes_decryptfinal', 8130)]>

In [85]:
for table_obj in TableDetailsModel.objects.filter(table_status=3):
    print(table_obj)

In [14]:
Task.objects.filter(status=0).count()

279

In [347]:
TableDetailsModel.objects.get(table_name = 'registry_reports')

<TableDetailsModel: registry_reports - northwest_metadata - 5771>

In [33]:
TableDetailsModel.objects.get(id = 3995)

<TableDetailsModel: lsm_actions - northwest_metadata - 3995>

In [286]:
[(v['is_phi'], v['de_identification_rule'], v['column_name']) for v in TableDetailsModel.objects.get(id = 6467).table_details_for_ui['columns_details']]

[(False, '', 'id'),
 (True, 'PATIENT_ID', 'contactId'),
 (True, 'PATIENT_ID', 'pid'),
 (True, 'MASK', 'name'),
 (False, '', 'relation'),
 (True, 'MASK', 'address'),
 (True, 'MASK', 'address2'),
 (True, 'MASK', 'city'),
 (False, '', 'state'),
 (True, 'ZIP_CODE', 'zipcode'),
 (True, 'MASK', 'homePhone'),
 (True, 'MASK', 'workPhone'),
 (False, '', 'delflag'),
 (False, '', 'contactType'),
 (True, 'DATE_OFFSET', 'timeVal'),
 (False, '', 'isFamilyMember'),
 (True, '', 'isEmergencyContact'),
 (True, 'PATIENT_DOB', 'dob'),
 (False, '', 'doi'),
 (True, 'MASK', 'cellPhone'),
 (False, None, 'nd_auto_increment_id')]

In [17]:
Task.all_objects.filter(arguments__table_id = 6360, failure_count__gt=0).count()

0

In [67]:
tables = ['progressnotes_decryptfinal', 'pnattachments', 'hcfa', 'surescript_eprescription', 'cpc_measure_config', 'dw_dim_psm', 'cqm_pcfmeasureconfig', 'ecliniforms', 'referral', 'tblportalcontactsdecrypted', 
          'trigger_codes_group', 'interactionnotes', 'itemkeys', 'lsm_actions', 'medsnotfoundlist', 'patientinfo']
len(tables)

16

In [357]:
Task.all_objects.filter(status=0).count()

5

In [348]:
Task.all_objects.filter(status=2, arguments__table_id=5771).count()

0

In [239]:
# Task.objects.filter(status=-1)[16].remarks

In [354]:
Task.objects.filter(status=-1,  arguments__table_id=5771).count()

0

In [350]:
Task.objects.filter(arguments__table_id=5771, failure_count__gt=0).count()

0

In [343]:
for t in Task.objects.filter(arguments__table_id=3166, status=-1):
    t.status = 0
    t.failure_count = 0
    t.soft_delete  = False
    t.save()

In [171]:
for t in Task.objects.filter(status__in=[0]):
    t.status = -1
    t.failure_count = 5
    t.remarks = {'reason': 'manual'}
    t.save()

In [20]:
# failed = []
# for i in range(Task.objects.filter(status=0).count()):
#     failed.append(Task.objects.filter(status=0)[i].arguments['table_id'])
# len(set(failed)), set(failed)

In [37]:
failed = []
for i in range(Task.objects.filter(status=-1).count()):
    failed.append(Task.objects.filter(status=-1)[i].arguments['table_id'])
len(set(failed)), set(failed)

(6, {1355, 1430, 3062, 4243, 5707, 8130})

In [152]:
# Task.objects.filter(status=-1).last().arguments['table_id']
# Task.objects.filter(status=-1).last().remarks

In [20]:
to =  TableDetailsModel.objects.get(table_name="hcfa")
a = {'conditions': [{'column_name': 'Id',
   'source_column': 'InvId',
   'reference_table': 'edi_invoice'}],
 'source_table': 'hcfa',
 'destination_column': 'EncounterId',
 'destination_column_type': 'encounter_id'}
to.table_details_for_ui['reference_mapping'] = a
to.save()


In [317]:
IgnoreRowsDeIdentificaiton.objects.filter(db_name="northwest_metadata", table_name="userprofile").count()

0

In [23]:
# Chain.objects.all().delete()

In [205]:
from django.conf import settings
settings.BATCH_SIZE_DURING_DE_IDENTIFICATION

1000

In [362]:
totalrun = 0
tables  = ['registry_reports']
for table in tables:
    tableobj = TableDetailsModel.objects.get(table_name=table)
    Task.objects.filter(arguments__table_id=tableobj.id).delete()
    # if tableobj.table_status in [2]:
    #     continue
    fix_reference_value(tableobj.id)
    tableobj.refresh_from_db()
    tasks, chain = create_deidentification_task(tableobj, True)
    totalrun += 1
    print(tableobj.rows_count)

2025-05-22 06:34:07,202 - deIdentification.nd_logger - INFO - inside marking as in progress if required
2025-05-22 06:34:07,218 - deIdentification.nd_logger - INFO - Table registry_reports dropped from the database.
2025-05-22 06:34:07,218 - deIdentification.nd_logger - INFO - Dropping ignore rows for registry_reports, northwest_metadata


[]
[]
22


(0, {})

In [100]:
for task in Task.all_objects.filter(status=-1, arguments__table_id=6360):
    task.failure_count = 0
    task.status = 0
    task.soft_delete = False
    task.save()

In [359]:
to = TableDetailsModel.objects.get(table_name="registry_reports")
# to.table_details_for_ui['reference_mapping'] = {'conditions': [{'column_name': 'ReportId',
#    'source_column': 'ReportId',
#    'reference_table': 'labdata'}],
#  'source_table': 'labdatadetail',
#  'destination_column': 'EncounterId',
#  'destination_column_type': 'encounter_id'}

# to.save()

In [361]:
to.table_details_for_ui = {'batch_size': 1000,
 'ignore_rows': {},
 'columns_details': [{'is_phi': False,
   'mask_value': '',
   'column_name': 'ReportId',
   'ignore_rows': {},
   'add_to_phi_table': False,
   'reference_mapping': {},
   'de_identification_rule': '',
   'column_name_for_phi_table': None},
  {'is_phi': True,
   'mask_value': '',
   'column_name': 'ReportName',
   'ignore_rows': {},
   'add_to_phi_table': False,
   'reference_mapping': {},
   'de_identification_rule': 'GENERIC_NOTES',
   'column_name_for_phi_table': None},
  {'is_phi': False,
   'mask_value': '',
   'column_name': 'UserId',
   'ignore_rows': {},
   'add_to_phi_table': False,
   'reference_mapping': {},
   'de_identification_rule': '',
   'column_name_for_phi_table': None},
  {'is_phi': False,
   'mask_value': '',
   'column_name': 'Scheduled',
   'ignore_rows': {},
   'add_to_phi_table': False,
   'reference_mapping': {},
   'de_identification_rule': '',
   'column_name_for_phi_table': None},
  {'is_phi': False,
   'mask_value': '',
   'column_name': 'FlowsheetId',
   'ignore_rows': {},
   'add_to_phi_table': False,
   'reference_mapping': {},
   'de_identification_rule': '',
   'column_name_for_phi_table': None},
  {'is_phi': False,
   'mask_value': '',
   'column_name': 'ProtocolId',
   'ignore_rows': {},
   'add_to_phi_table': False,
   'reference_mapping': {},
   'de_identification_rule': '',
   'column_name_for_phi_table': None},
  {'is_phi': False,
   'mask_value': '',
   'column_name': 'LastRun',
   'ignore_rows': {},
   'add_to_phi_table': False,
   'reference_mapping': {},
   'de_identification_rule': '',
   'column_name_for_phi_table': None},
  {'is_phi': False,
   'mask_value': '',
   'column_name': 'Status',
   'ignore_rows': {},
   'add_to_phi_table': False,
   'reference_mapping': {},
   'de_identification_rule': '',
   'column_name_for_phi_table': None},
  {'is_phi': False,
   'mask_value': '',
   'column_name': 'Type',
   'ignore_rows': {},
   'add_to_phi_table': False,
   'reference_mapping': {},
   'de_identification_rule': '',
   'column_name_for_phi_table': None},
  {'is_phi': False,
   'mask_value': '',
   'column_name': 'ProjectId',
   'ignore_rows': {},
   'add_to_phi_table': False,
   'reference_mapping': {},
   'de_identification_rule': '',
   'column_name_for_phi_table': None},
  {'is_phi': False,
   'mask_value': '',
   'column_name': 'deleteflag',
   'ignore_rows': {},
   'add_to_phi_table': False,
   'reference_mapping': {},
   'de_identification_rule': '',
   'column_name_for_phi_table': None},
  {'is_phi': True,
   'mask_value': '',
   'column_name': 'reportcriteria',
   'ignore_rows': {},
   'add_to_phi_table': False,
   'reference_mapping': {},
   'de_identification_rule': 'GENERIC_NOTES',
   'column_name_for_phi_table': None},
  {'is_phi': False,
   'mask_value': '',
   'column_name': 'OrderSetId',
   'ignore_rows': {},
   'add_to_phi_table': False,
   'reference_mapping': {},
   'de_identification_rule': '',
   'column_name_for_phi_table': None},
  {'is_phi': False,
   'mask_value': '',
   'column_name': 'cdss',
   'ignore_rows': {},
   'add_to_phi_table': False,
   'reference_mapping': {},
   'de_identification_rule': '',
   'column_name_for_phi_table': None},
  {'is_phi': False,
   'mask_value': '',
   'column_name': 'conditionflag',
   'ignore_rows': {},
   'add_to_phi_table': False,
   'reference_mapping': {},
   'de_identification_rule': '',
   'column_name_for_phi_table': None},
  {'is_phi': False,
   'mask_value': '',
   'column_name': 'DatePrompt',
   'ignore_rows': {},
   'add_to_phi_table': False,
   'reference_mapping': {},
   'de_identification_rule': '',
   'column_name_for_phi_table': None},
  {'is_phi': False,
   'mask_value': '',
   'column_name': 'remindertype',
   'ignore_rows': {},
   'add_to_phi_table': False,
   'reference_mapping': {},
   'de_identification_rule': '',
   'column_name_for_phi_table': None},
  {'is_phi': False,
   'mask_value': '',
   'column_name': 'ReportMPIId',
   'ignore_rows': {},
   'add_to_phi_table': False,
   'reference_mapping': {},
   'de_identification_rule': '',
   'column_name_for_phi_table': None},
  {'is_phi': False,
   'column_name': 'nd_auto_increment_id',
   'ignore_rows': {},
   'add_to_phi_table': False,
   'reference_mapping': {},
   'de_identification_rule': None,
   'column_name_for_phi_table': None}],
 'reference_mapping': {},
 'reference_enc_id_column': None,
 'reference_patient_id_column': None}

to.save()

In [ ]:
ImmId